# MMA / UFC fighter pose — YOLO training (Colab + Google Drive)

1. Install **Ultralytics** and dependencies  
2. **Mount Google Drive**  
3. Point to your **YOLO pose dataset** on Drive (folder containing `data.yaml`, `train/`, `valid/` or `val/`, etc.)  
4. **Train** on Colab GPU (checkpoints under `/content` for speed)  
5. Validate, **MAE/RMSE**, **§8** Matplotlib PNGs + **§8b** Plotly curves  
6. Save **best.pt**, metrics JSON, and plots to a folder on **Drive**  
7. **§7b** (optional): interactive **tactic / punch-kick** preview (Plotly timeline + real frame strips)  
8. **§8b** (optional): interactive **training curves** (`results.csv` → multi-row Plotly + `training_curves_plotly.html`)  
9. Optional **§10–§12**: clip manifest, **LSTM**, **MMAction2** export  

**Before running:** upload your dataset to Drive (same layout as Ultralytics: `data.yaml` next to `train/images`, `train/labels`, …).

**Runtime:** `Runtime` → `Change runtime type` → **GPU**.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics pandas matplotlib numpy pyyaml opencv-python-headless tqdm torch scikit-learn plotly

## 2. Configuration — edit Drive paths

- **`DRIVE_DATASET_DIR`**: folder that contains **`data.yaml`** and your `train` / `valid` (or `val`) splits.  
- **`DRIVE_OUTPUT_DIR`**: where to save each run (`best.pt`, `metrics_summary.json`, `plots/`, `train_run/`).

In [ ]:
from __future__ import annotations

import json
import shutil
from datetime import datetime
from pathlib import Path

# --- Google Drive paths (after mount, /content/drive/MyDrive/...) ---
DRIVE_DATASET_DIR = Path(
    "/content/drive/MyDrive/mma-fighter-pose-estimation-dataset"
)
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/UFC-PoseEstimation-runs")

# Training (lower batch or use yolo11n-pose.pt on small GPUs)
MODEL_NAME = "yolo11s-pose.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
RUN_NAME = "mma_pose_colab"

# Fast local disk for Ultralytics training cache
WORK_ROOT = Path("/content/mma_pose_work")
TRAIN_PROJECT = WORK_ROOT / "ultra_runs"

## 3. Mount Google Drive

Authorize when prompted so `/content/drive/MyDrive/...` paths work.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Dataset & output paths

Checks that `data.yaml` exists and that each split’s `images` folder has a sibling `labels` folder (case-insensitive names, e.g. `Train`/`Images`).

In [ ]:
import yaml


def _ci_child(parent: Path, logical_name: str) -> Path | None:
    """Case-insensitive child dir (Train vs train, Labels vs labels)."""
    if not parent.is_dir():
        return None
    want = logical_name.lower()
    for c in parent.iterdir():
        if c.is_dir() and c.name.lower() == want:
            return c
    return None


def verify_yolo_dataset(data_yaml: Path) -> None:
    root = data_yaml.parent.resolve()
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    dr = root
    if cfg.get("path"):
        p = Path(str(cfg["path"]))
        dr = p.resolve() if p.is_absolute() else (root / p).resolve()
    else:
        dr = root
    for key in ("train", "val", "valid", "test"):
        if key not in cfg or cfg[key] in (None, ""):
            continue
        rel = Path(str(cfg[key]))
        full = rel.resolve() if rel.is_absolute() else (dr / rel).resolve()
        assert full.is_dir(), f"Missing split images folder ({key}): {full}"
        if full.name.lower() == "images":
            labels = _ci_child(full.parent, "labels")
            assert labels and labels.is_dir(), f"Missing labels folder next to {full}"
    print("Dataset layout OK:", data_yaml)


DATASET_DIR = DRIVE_DATASET_DIR.expanduser().resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.is_file(), (
    f"Missing {DATA_YAML}\n"
    "Set DRIVE_DATASET_DIR to the folder that contains data.yaml (and train/, valid/, …)."
)

OUTPUT_DIR = DRIVE_OUTPUT_DIR.expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

verify_yolo_dataset(DATA_YAML)
print("DATASET_DIR:", DATASET_DIR)
print("DATA_YAML:", DATA_YAML)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 5. Train YOLO pose

Weights and logs go under `/content/mma_pose_work/ultra_runs` (fast disk). The next section copies weights, plots, and JSON to `OUTPUT_DIR`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
)

RUN_DIR = TRAIN_PROJECT / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.is_file(), f"Missing {BEST_PT}"
print("Training done. best.pt:", BEST_PT)

## 6. Official validation metrics (Ultralytics)

In [ ]:
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ, split="val", plots=True)

val_dict = val_metrics.results_dict if hasattr(val_metrics, "results_dict") else dict(val_metrics)
print("Validation summary (excerpt):")
for k in sorted(val_dict.keys()):
    if "map" in k.lower() or "precision" in k.lower() or "recall" in k.lower() or "loss" in k.lower():
        print(f"  {k}: {val_dict[k]}")

## 7. Helper: MAE / RMSE vs ground truth (val split)

Matches each GT box to the highest-IoU prediction, then compares **visible** keypoints (`v > 0`) in **normalized** image coordinates.

In [ ]:
import re

import numpy as np
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

K = 17
KPT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]


def _dataset_root(data_yaml: Path) -> Path:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    p = cfg.get("path")
    if p:
        root = Path(p)
        return root if root.is_absolute() else (data_yaml.parent / root).resolve()
    return data_yaml.parent.resolve()


def _resolve_split_dir(root: Path, key: str, data_yaml: Path) -> Path:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    rel = cfg.get(key) or cfg.get("val") or cfg.get("valid")
    if not rel:
        raise ValueError(f"No val/valid path in {data_yaml}")
    rel = str(rel)
    p = Path(rel)
    return p if p.is_absolute() else (root / p).resolve()


def cxcywhn_to_xyxyn(cx, cy, w, h):
    x1, y1 = cx - w / 2.0, cy - h / 2.0
    x2, y2 = cx + w / 2.0, cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float64)


def iou_xyxyn(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def read_pose_labels(label_path: Path) -> list[dict]:
    """Each dict: bbox (cx,cy,w,h), xy (17,2), vis (17,) int."""
    if not label_path.is_file():
        return []
    out = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5 + K * 3:
            continue
        cls = int(float(parts[0]))
        cx, cy, w, h = map(float, parts[1:5])
        rest = list(map(float, parts[5 : 5 + K * 3]))
        xy = np.array(rest[0::3], dtype=np.float64).reshape(-1, 1)
        yy = np.array(rest[1::3], dtype=np.float64).reshape(-1, 1)
        vis = np.array(rest[2::3], dtype=np.int64)
        xy = np.hstack([xy, yy])
        out.append({"cls": cls, "bbox": (cx, cy, w, h), "xy": xy, "vis": vis})
    return out


def evaluate_keypoint_errors(model: YOLO, data_yaml: Path, max_images: int | None = None):
    root = _dataset_root(data_yaml)
    val_img_root = _resolve_split_dir(root, "val", data_yaml)
    lbl_root = val_img_root.parent / "labels"
    assert val_img_root.is_dir(), val_img_root
    assert lbl_root.is_dir(), lbl_root

    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    images = sorted([p for p in val_img_root.rglob("*") if p.suffix.lower() in exts])
    if max_images:
        images = images[:max_images]

    per_k_sq = [[] for _ in range(K)]
    per_k_abs = [[] for _ in range(K)]
    all_sq = []
    all_abs = []
    n_matched_instances = 0

    for img_path in tqdm(images, desc="MAE/RMSE val"):
        stem = img_path.stem
        if stem.endswith(".rf"):
            m = re.match(r"^(.*)\.rf\.[a-f0-9]+$", stem)
            stem_alt = m.group(1) if m else stem
        else:
            stem_alt = stem
        lbl_path = lbl_root / f"{stem}.txt"
        if not lbl_path.is_file():
            lbl_path = lbl_root / f"{stem_alt}.txt"
        gts = read_pose_labels(lbl_path)
        if not gts:
            continue

        results = model.predict(source=str(img_path), imgsz=IMGSZ, verbose=False)
        if not results:
            continue
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0 or r.keypoints is None or len(r.keypoints) == 0:
            continue

        pred_xyxyn = r.boxes.xyxyn.cpu().numpy()
        pred_xyn = r.keypoints.xyn.cpu().numpy()

        for gt in gts:
            gt_xyxy = cxcywhn_to_xyxyn(*gt["bbox"])
            best_j = -1
            best_iou = 0.0
            for j in range(pred_xyxyn.shape[0]):
                iou = iou_xyxyn(gt_xyxy, pred_xyxyn[j])
                if iou > best_iou:
                    best_iou = iou
                    best_j = j
            if best_j < 0 or best_iou < 0.1:
                continue
            n_matched_instances += 1
            px = pred_xyn[best_j]
            gx = gt["xy"]
            vis = gt["vis"]
            for ki in range(K):
                if vis[ki] <= 0:
                    continue
                d = np.linalg.norm(px[ki] - gx[ki])
                per_k_sq[ki].append(d * d)
                per_k_abs[ki].append(d)
                all_sq.append(d * d)
                all_abs.append(d)

    def rmse(xs):
        return float(np.sqrt(np.mean(xs))) if xs else float("nan")

    def mae(xs):
        return float(np.mean(xs)) if xs else float("nan")

    per_k = {
        KPT_NAMES[i]: {"mae": mae(per_k_abs[i]), "rmse": rmse(per_k_sq[i]), "n": len(per_k_abs[i])}
        for i in range(K)
    }
    summary = {
        "overall_mae_normalized": mae(all_abs),
        "overall_rmse_normalized": rmse(all_sq),
        "n_keypoint_comparisons": len(all_abs),
        "n_matched_instances": n_matched_instances,
        "note": "Errors in normalized [0,1] image coordinates; multiply by image side for rough pixels.",
        "per_keypoint": per_k,
    }
    return summary


kp_summary = evaluate_keypoint_errors(best_model, DATA_YAML, max_images=None)
print(json.dumps({k: v for k, v in kp_summary.items() if k != "per_keypoint"}, indent=2))

## 7b. Interactive tactic preview (real frames + heuristic combos)

Uses **`best_model`** predictions on the **val** split, same strike heuristics as `webcam_pose.analyze_recording_keypoints`, then clusters nearby peaks into **`tactic_proxy`** rows (e.g. `multi_punch_proxy`, `punch_kick_chain_proxy`).

**Output:** `OUTPUT_DIR/tactics_interactive.html` — **zoom/pan/hover** Plotly timeline + **JPEG-embedded frame strips** (each row = one combo; **thumbnails left→right in time**, skeleton overlay). Strips use embedded data-URI JPEGs (standard HTML images, not Plotly) so they render in Colab.

**Setup:** clone or upload this repo so `YOLO_POSE_ROOT` points at `yoloPose` (must contain `tools/tactic_interactive_gallery.py`, `val_pose_insights.py`, `webcam_pose.py`).


In [ ]:
# §7b — Interactive Plotly gallery (tactics / punch sequences on real val images)
import sys
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

YOLO_POSE_ROOT = Path("/content/yoloPose")
sys.path.insert(0, str(YOLO_POSE_ROOT))

try:
    import plotly.graph_objects as go  # noqa: F401
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])

gallery_py = YOLO_POSE_ROOT / "tools" / "tactic_interactive_gallery.py"
if not gallery_py.is_file():
    print("Upload/clone repo to YOLO_POSE_ROOT so tools/tactic_interactive_gallery.py exists.")
else:
    sys.path.insert(0, str(YOLO_POSE_ROOT / "tools"))
    from tactic_interactive_gallery import build_interactive_tactic_report

    _dev = None
    if torch is not None and torch.cuda.is_available():
        _dev = "0"
    out = OUTPUT_DIR / "tactics_interactive.html"
    build_interactive_tactic_report(
        best_model,
        Path(DATA_YAML),
        out,
        split="val",
        imgsz=int(IMGSZ),
        device=_dev,
        max_fights=10,
        max_frames_per_fight=120,
        max_predict_total=500,
        max_examples_per_tactic=4,
        sequence_len=5,
        kpt_conf_draw=0.35,
    )
    from IPython.display import HTML, display

    display(HTML(f'<a href="{out}" target="_blank">Open tactics_interactive.html</a> (or Files panel on Drive)'))
    print("Wrote", out)


## 8. Plots (10+ figures)

Run this section after training and §7 (keypoint metrics).

- **Matplotlib (this cell + next):** loss grids from `results.csv`, optional mAP / P–R / LR curves, **per-keypoint MAE & RMSE**, error histogram, sample val overlays — all saved as **PNGs** under `OUTPUT_DIR/plots/`.
- **§8b Plotly:** same run as **interactive** multi-row charts + **`training_curves_plotly.html`** (open in Colab if inline charts lag).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        plt.style.use("default")

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

results_csv = RUN_DIR / "results.csv"
df = pd.read_csv(results_csv) if results_csv.is_file() else None

fig_n = 0


def save_fig(name: str):
    global fig_n
    fig_n += 1
    path = PLOTS_DIR / f"{fig_n:02d}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved", path)


if df is not None and len(df) > 0:
    epochs = np.arange(1, len(df) + 1)

    def col(*candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    # Fig 1–4: core losses (4 subplots = 4 visualizations)
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    ax = axes.ravel()
    pairs = [
        (col("train/box_loss", "train/box_loss"), "Train box loss"),
        (col("train/pose_loss", "train/pose_loss"), "Train pose loss"),
        (col("val/box_loss"), "Val box loss"),
        (col("val/pose_loss"), "Val pose loss"),
    ]
    for i, (cname, title) in enumerate(pairs):
        if cname and cname in df.columns:
            ax[i].plot(epochs, df[cname].values)
            ax[i].set_title(title)
            ax[i].set_xlabel("epoch")
            ax[i].grid(True, alpha=0.3)
        else:
            ax[i].set_visible(False)
    plt.suptitle("Training / validation losses", y=1.02)
    save_fig("losses_box_pose")

    # Fig 5: mAP-style metrics if present
    metric_cols = [c for c in df.columns if "mAP" in c or "map" in c.lower()]
    metric_cols = [c for c in metric_cols if "/" in c][:6]
    if metric_cols:
        fig, axes = plt.subplots(1, len(metric_cols), figsize=(4 * len(metric_cols), 3.5))
        if len(metric_cols) == 1:
            axes = [axes]
        for ax, c in zip(axes, metric_cols):
            ax.plot(epochs, df[c].values)
            ax.set_title(c.replace("metrics/", ""))
            ax.set_xlabel("epoch")
            ax.grid(True, alpha=0.3)
        plt.suptitle("mAP / metrics vs epoch", y=1.05)
        save_fig("map_metrics")

    # Fig 6: precision / recall
    pr_cols = [c for c in df.columns if "precision" in c.lower() or "recall" in c.lower()][:4]
    if pr_cols:
        fig, axes = plt.subplots(2, 2, figsize=(10, 7))
        axes = axes.ravel()
        for i, c in enumerate(pr_cols):
            axes[i].plot(epochs, df[c].values, label=c)
            axes[i].set_title(c)
            axes[i].set_xlabel("epoch")
            axes[i].grid(True, alpha=0.3)
        for j in range(len(pr_cols), 4):
            axes[j].set_visible(False)
        plt.suptitle("Precision / recall", y=1.02)
        save_fig("precision_recall")

    # Fig 7: learning rate
    lr_c = col("lr/pg0", "train/lr0", "lr0")
    if lr_c and lr_c in df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(epochs, df[lr_c].values)
        plt.title("Learning rate schedule")
        plt.xlabel("epoch")
        plt.grid(True, alpha=0.3)
        save_fig("learning_rate")

    # Fig 8: combined train total-ish proxy
    tcols = [c for c in [col("train/box_loss"), col("train/pose_loss")] if c and c in df.columns]
    if tcols:
        plt.figure(figsize=(7, 4))
        for c in tcols:
            plt.plot(epochs, df[c].values, label=c)
        plt.legend()
        plt.title("Train losses overlay")
        plt.xlabel("epoch")
        plt.grid(True, alpha=0.3)
        save_fig("train_loss_overlay")
else:
    print("No results.csv found at", results_csv)

In [ ]:
# Fig 9–10: MAE / RMSE per keypoint
pk = kp_summary["per_keypoint"]
names = list(pk.keys())
maes = [pk[n]["mae"] for n in names]
rmses = [pk[n]["rmse"] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(K), maes, color="steelblue")
axes[0].set_xticks(range(K))
axes[0].set_xticklabels(names, rotation=60, ha="right")
axes[0].set_ylabel("MAE (normalized)")
axes[0].set_title("Per-keypoint MAE (visible GT)")
axes[0].grid(True, axis="y", alpha=0.3)
axes[1].bar(range(K), rmses, color="darkorange")
axes[1].set_xticks(range(K))
axes[1].set_xticklabels(names, rotation=60, ha="right")
axes[1].set_ylabel("RMSE (normalized)")
axes[1].set_title("Per-keypoint RMSE (visible GT)")
axes[1].grid(True, axis="y", alpha=0.3)
plt.suptitle("Keypoint errors on val", y=1.02)
save_fig("per_keypoint_mae_rmse")

# Fig 11: histogram of all point errors
root = _dataset_root(DATA_YAML)
val_img_root = _resolve_split_dir(root, "val", DATA_YAML)
lbl_root = val_img_root.parent / "labels"
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
val_images = sorted([p for p in val_img_root.rglob("*") if p.suffix.lower() in exts])[:400]
hist_errors = []
for img_path in tqdm(val_images, desc="Histogram errors"):
    stem = img_path.stem
    lbl_path = lbl_root / f"{stem}.txt"
    if not lbl_path.is_file():
        continue
    gts = read_pose_labels(lbl_path)
    if not gts:
        continue
    results = best_model.predict(source=str(img_path), imgsz=IMGSZ, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        continue
    r = results[0]
    pred_xyxyn = r.boxes.xyxyn.cpu().numpy()
    pred_xyn = r.keypoints.xyn.cpu().numpy()
    for gt in gts:
        gt_xyxy = cxcywhn_to_xyxyn(*gt["bbox"])
        best_j = int(np.argmax([iou_xyxyn(gt_xyxy, pred_xyxyn[j]) for j in range(len(pred_xyxyn))]))
        if iou_xyxyn(gt_xyxy, pred_xyxyn[best_j]) < 0.1:
            continue
        px = pred_xyn[best_j]
        for ki in range(K):
            if gt["vis"][ki] <= 0:
                continue
            hist_errors.append(float(np.linalg.norm(px[ki] - gt["xy"][ki])))

plt.figure(figsize=(7, 4))
if hist_errors:
    plt.hist(hist_errors, bins=40, color="seagreen", edgecolor="white")
else:
    plt.text(0.5, 0.5, "No errors collected", ha="center", va="center")
plt.xlabel("L2 error (normalized)")
plt.ylabel("count")
plt.title("Distribution of per-keypoint errors (val, visible GT)")
plt.grid(True, alpha=0.3)
save_fig("error_histogram")

# Fig 12–15: sample prediction panels
import cv2

if len(val_images) >= 4:
    step = max(1, len(val_images) // 8)
    sample_paths = val_images[::step][:8]
else:
    sample_paths = list(val_images)
if len(sample_paths) < 4 and val_images:
    sample_paths = val_images[:4]

for si, sp in enumerate(sample_paths[:4]):
    res = best_model.predict(source=str(sp), imgsz=IMGSZ, verbose=False)[0]
    im = res.plot()
    im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if len(im.shape) == 3 and im.shape[2] == 3 else im
    plt.figure(figsize=(8, 8))
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title(f"Val prediction overlay: {sp.name}")
    save_fig(f"sample_pred_{si+1}")

print(f"Total figure files saved under: {PLOTS_DIR} (count ~{fig_n})")

## 8b. Interactive training curves (Plotly)

Multi-row chart: **losses**, **mAP / P / R** (if present), **learning rate**. Hover and zoom; also saves **`plots/training_curves_plotly.html`** (reliable in Colab when inline output is slow).


In [ ]:
# §8b — Interactive training curves (Plotly): multi-row dashboard + HTML export
from pathlib import Path

import pandas as pd

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots

pio.renderers.default = "colab"

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
results_csv = RUN_DIR / "results.csv"

if not results_csv.is_file():
    print("No results.csv yet — run training (§5) first.")
else:
    df = pd.read_csv(results_csv)
    xcol = "epoch" if "epoch" in df.columns else df.columns[0]
    num_cols = [
        c
        for c in df.columns
        if c != xcol and pd.api.types.is_numeric_dtype(df[c])
    ]

    loss_cols = [c for c in num_cols if "loss" in c.lower()]
    metric_cols = [
        c
        for c in num_cols
        if c not in loss_cols
        and any(k in c.lower() for k in ("map", "precision", "recall", "f1"))
    ][:10]
    lr_cols = [c for c in num_cols if "lr" in c.lower()]

    groups = []
    if loss_cols:
        groups.append(("Losses", loss_cols))
    if metric_cols:
        groups.append(("mAP / precision / recall", metric_cols))
    if lr_cols:
        groups.append(("Learning rate", lr_cols))
    if not groups:
        groups = [("Numeric columns", num_cols[: min(12, len(num_cols))])]

    R = len(groups)
    fig = make_subplots(
        rows=R,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.07,
        subplot_titles=tuple(t[0] for t in groups),
    )

    for ri, (_title, cols) in enumerate(groups, start=1):
        for c in cols:
            fig.add_trace(
                go.Scatter(
                    x=df[xcol],
                    y=df[c],
                    mode="lines+markers",
                    name=c,
                    legendgroup="r%d" % ri,
                    hovertemplate="%{y:.5f}<extra></extra>",
                ),
                row=ri,
                col=1,
            )
        fig.update_yaxes(title_text="value", row=ri, col=1)

    fig.update_xaxes(title_text=xcol, row=R, col=1)
    fig.update_layout(
        title="Training / validation curves (interactive)",
        template="plotly_dark",
        height=min(260 * R + 120, 1400),
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(t=80),
    )
    fig.show()

    html_path = PLOTS_DIR / "training_curves_plotly.html"
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    print("Wrote", html_path)

    try:
        from IPython.display import HTML, display

        display(HTML('<a href="' + str(html_path) + '" target="_blank">Open training_curves_plotly.html</a> (full-window)'))
    except Exception:
        pass


## 9. Save model + metrics (`OUTPUT_DIR` on Drive)

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_run = OUTPUT_DIR / f"run_{stamp}"
out_run.mkdir(parents=True, exist_ok=True)

results_csv = RUN_DIR / "results.csv"

shutil.copy2(BEST_PT, out_run / "best.pt")
if results_csv.is_file():
    shutil.copy2(results_csv, out_run / "results.csv")

# Optional: full training folder (weights, args, curves)
shutil.copytree(RUN_DIR, out_run / "train_run", dirs_exist_ok=True)

def _json_scalar(v):
    try:
        if hasattr(v, "item"):
            return float(v.item())
        return float(v)
    except (TypeError, ValueError):
        return str(v)


metrics_bundle = {
    "timestamp": stamp,
    "data_yaml": str(DATA_YAML),
    "model_base": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "ultralytics_val": {k: _json_scalar(v) for k, v in val_dict.items()},
    "keypoint_mae_rmse": kp_summary,
}

with open(out_run / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_bundle, f, indent=2, default=str)

print("Saved to:", out_run)
print(" - best.pt")
print(" - metrics_summary.json")
print(" - train_run/ (full Ultralytics export)")
print(" - ../plots/ (PNG figures)")

## 10. Action clips: manifest + `.npy` sequences (for LSTM / MMAction2)

**You need class labels** (e.g. `jab`, `hook`, `other`) per clip. See `schemas/action_clip_manifest.example.csv` in the repo.

**Columns:** `clip_id`, `label`, `npy_path`, `split` (`train` or `val`), optional `img_h`, `img_w`.

**Option A — Build from val GT (this repo):** clone or upload `yoloPose/tools/` to Colab, then run the next cell (sets paths under `OUTPUT_DIR/action_clips/`).

**Option B — Bring your own:** save each clip as `(T, 17, 2)` or `(T, 17, 3)` float32 `.npy` and write `manifest.csv` yourself.

Normalize keypoints (mid-hip center, shoulder scale) when exporting to MMAction2 (`--normalize` on the export script) or inside the LSTM cell if you prefer.


In [ ]:
# §10 — Build manifest + sequences from val GT (optional; requires repo tools on Colab)
import subprocess
import sys
from pathlib import Path

# Set to your cloned yoloPose root on Colab, or upload the `tools/` folder and point here:
YOLO_POSE_ROOT = Path("/content/yoloPose")  # e.g. git clone ... into /content/yoloPose
ACTION_OUT = OUTPUT_DIR / "action_clips"
MANIFEST_CSV = ACTION_OUT / "manifest.csv"

build_py = YOLO_POSE_ROOT / "tools" / "build_manifest_from_val_gt.py"
if build_py.is_file():
    ACTION_OUT.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            str(build_py),
            "--data-yaml",
            str(DATA_YAML),
            "--out-dir",
            str(ACTION_OUT),
            "--min-frames",
            "8",
        ]
    )
    # Template uses label "unlabeled" — copy/edit before training:
    tpl = ACTION_OUT / "manifest_template.csv"
    if tpl.is_file() and not MANIFEST_CSV.is_file():
        import shutil
        shutil.copy(tpl, MANIFEST_CSV)
        print("Copied manifest_template.csv -> manifest.csv — EDIT `label` column, then run §11.")
    else:
        print("Wrote template; set MANIFEST_CSV to your labeled manifest.")
else:
    print("Skip: clone repo to YOLO_POSE_ROOT or upload tools/. Build manifest.csv manually (see §10 markdown).")
print("MANIFEST_CSV path:", MANIFEST_CSV)


## 11. LSTM baseline (sequence classification)

Trains a small 2-layer LSTM on padded clips `(T_fix, 17*2)`. Rows with `label` `unlabeled` or empty are skipped. Uses `split` column from CSV for train/val.


In [ ]:
# §11 — LSTM baseline (PyTorch)
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset

MANIFEST_CSV = OUTPUT_DIR / "action_clips" / "manifest.csv"
T_FIX = 64
BATCH = 16
EPOCHS = 15
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not MANIFEST_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {MANIFEST_CSV}. Run §10 or create CSV (see schemas/action_clip_manifest.example.csv)."
    )


def _load_rows():
    rows = []
    with open(MANIFEST_CSV, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            lab = row.get("label", "").strip()
            if not lab or lab.lower() == "unlabeled":
                continue
            row["npy_path"] = str(Path(row["npy_path"]).expanduser().resolve())
            rows.append(row)
    return rows


all_rows = _load_rows()
if len(all_rows) < 2:
    raise RuntimeError("Need at least 2 labeled rows in manifest.csv (not unlabeled).")

labels = sorted({r["label"] for r in all_rows})
label_to_id = {lb: i for i, lb in enumerate(labels)}
id_to_label = {i: lb for lb, i in label_to_id.items()}
train_rows = [r for r in all_rows if r.get("split", "train").lower() == "train"]
val_rows = [r for r in all_rows if r.get("split", "train").lower() == "val"]
if not train_rows or not val_rows:
    raise RuntimeError("Manifest needs at least one train and one val row (after filtering unlabeled).")


class ClipDataset(Dataset):
    def __init__(self, rows, t_fix: int):
        self.rows = rows
        self.t_fix = t_fix

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        a = np.load(r["npy_path"])
        if a.ndim != 3 or a.shape[1] != 17:
            raise ValueError(r["npy_path"], a.shape)
        xy = a[:, :, :2].astype(np.float32)
        t = xy.shape[0]
        if t >= self.t_fix:
            xy = xy[: self.t_fix]
        else:
            pad = np.zeros((self.t_fix - t, 17, 2), dtype=np.float32)
            xy = np.concatenate([xy, pad], axis=0)
        x = torch.from_numpy(xy.reshape(self.t_fix, -1))
        y = label_to_id[r["label"]]
        return x, y


class LSTMClf(nn.Module):
    def __init__(self, in_dim: int, n_cls: int, hidden: int = 128):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden, n_cls)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


in_dim = 17 * 2
n_cls = len(labels)
model = LSTMClf(in_dim, n_cls).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

train_dl = DataLoader(ClipDataset(train_rows, T_FIX), batch_size=BATCH, shuffle=True)
val_dl = DataLoader(ClipDataset(val_rows, T_FIX), batch_size=BATCH)

hist = {"tr": [], "va": []}
for ep in range(EPOCHS):
    model.train()
    tl = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        tl += loss.item() * x.size(0)
    tl /= len(train_dl.dataset)
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    va = correct / max(total, 1)
    hist["tr"].append(tl)
    hist["va"].append(va)
    print(f"epoch {ep+1}/{EPOCHS}  train_loss={tl:.4f}  val_acc={va:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(hist["tr"], label="train_loss")
plt.plot(hist["va"], label="val_acc")
plt.legend()
plt.xlabel("epoch")
plt.show()

# Confusion matrix on val
model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in val_dl:
        x = x.to(DEVICE)
        pr = model(x).argmax(dim=1).cpu().numpy()
        ys.append(y.numpy())
        ps.append(pr)
ys = np.concatenate(ys)
ps = np.concatenate(ps)
print(classification_report(ys, ps, target_names=[id_to_label[i] for i in range(n_cls)]))
cm = confusion_matrix(ys, ps)
print("confusion_matrix:\n", cm)

ckpt = OUTPUT_DIR / "lstm_action_best.pt"
torch.save({"model": model.state_dict(), "label_to_id": label_to_id, "t_fix": T_FIX}, ckpt)
print("Saved", ckpt)


## 12. MMAction2 skeleton pickle + ST-GCN train

**Version pins:** PyTorch, CUDA, MMCV, and MMAction2 must match. Follow the [official install guide](https://mmaction2.readthedocs.io/en/latest/get_started/installation.html) and Colab’s current CUDA.

**Steps:** (1) `openmim` + `mim install mmcv mmaction2` (2) clone `open-mmlab/mmaction2` (3) copy `configs/mmaction2/stgcn_coco17_2d.py` from this repo into `mmaction2/configs/skeleton/stgcn/` (4) copy the pickle to `mmaction2/data/skeleton/mmaction_custom.pkl` (5) set `custom_num_classes` in the config (6) `mim train mmaction2 stgcn_coco17_2d.py` from the **mmaction2 repo root**.

Annotation format reference: [Preparing Skeleton Dataset](https://mmaction2.readthedocs.io/en/latest/dataset_zoo/skeleton.html#the-format-of-annotations).


In [ ]:
# §12 — Export MMAction2 pickle + example train command (run in Colab after mim install)
import subprocess
import sys
from pathlib import Path

YOLO_POSE_ROOT = Path("/content/yoloPose")
MANIFEST_CSV = OUTPUT_DIR / "action_clips" / "manifest.csv"
MMACTION_DATA = Path("/content/mmaction2/data/skeleton")
export_py = YOLO_POSE_ROOT / "tools" / "export_mmaction2_skeleton.py"

if not MANIFEST_CSV.is_file():
    print("Missing manifest; complete §10–§11 labeling first.")
elif export_py.is_file():
    MMACTION_DATA.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            str(export_py),
            "--manifest",
            str(MANIFEST_CSV),
            "--out-dir",
            str(MMACTION_DATA),
            "--normalize",
            "--pkl-name",
            "mmaction_custom.pkl",
        ]
    )
    print("Pickle written under", MMACTION_DATA)
    print("Copy to mmaction2 clone if different path, then:")
    print("  cd /content/mmaction2 && mim train mmaction2 configs/skeleton/stgcn/stgcn_coco17_2d.py")
else:
    print("Upload yoloPose tools/ or set YOLO_POSE_ROOT; run export manually:")
    print("  python tools/export_mmaction2_skeleton.py --manifest ... --out-dir ... --normalize")

# Example installs (uncomment after checking MMCV + torch CUDA matrix for your Colab runtime):
# !pip install -U openmim
# !mim install mmengine "mmcv>=2.0.0" mmaction2


## Notes

- **Ultralytics** pose metrics are mainly **OKS / mAP** style; **MAE/RMSE** here are extra, in **normalized** coordinates.
- If **IoU matching** fails often, lower the threshold in `evaluate_keypoint_errors` or inspect bbox alignment.
- **`model.val(..., plots=True)`** writes extra plots into the local run folder; those are copied under `train_run/` in the final save step.
- Training uses **`/content`** for speed; only **results** are written to **Drive** (`OUTPUT_DIR`).
- **`data.yaml`** should use paths that work on Linux (or rely on `path:` + relative splits). Folder names like `Train`/`Images` are OK; verification is case-insensitive.